# Out-of-Sample Portfolio Allocation

Compares maximum Sharpe, HRP and MST portfolios under chronological training and test periods.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
Portfolio Optimization Strategies: Max Sharpe, HRP, and MST
============================================================

This script implements three portfolio optimization strategies:
1. Maximum Sharpe Ratio (Mean-Variance Optimization)
2. Hierarchical Risk Parity (HRP)
3. Minimum Spanning Tree (MST)

Trading Flow:
- Signal generated at Close[t]
- Execution at Open[t+1]
- Portfolio valued at Open[t+1]
- Returns calculated as Open[t+2]/Open[t+1] - 1

Author: Trading Strategy Implementation
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")


class Config:
    """Configuration class to hold all adjustable parameters"""

    TICKERS = ["SPY", "EFA", "EEM", "VGK", "VPL", "EWJ", "FXI", "GLD", "BTC-USD"]
    BENCHMARK_TICKER = "URTH"
    IS_START = "2013-01-01"
    IS_END = "2019-12-31"
    OOS_START = "2020-01-01"
    OOS_END = "2023-12-31"
    COMMISSION_RATE = 0.001
    INITIAL_CAPITAL = 100000
    REBALANCE_FREQUENCY = "Q"
    RISK_FREE_RATE = 0.02
    MAX_WEIGHT = 0.3
    MIN_WEIGHT = 0.0
    LOOKBACK_DAYS = 252
    FIGSIZE = (15, 10)
    DPI = 100
    STYLE = "seaborn-v0_8-darkgrid"


class DataFetcher:
    """Handles all data fetching from Yahoo Finance"""

    def fetch_historical_data(self, ticker, start_date, end_date):
        """
        Fetch historical OHLCV data from Yahoo Finance

        Parameters:
        -----------
        ticker : str
            Stock/ETF ticker symbol
        start_date : str
            Start date in 'YYYY-MM-DD' format
        end_date : str
            End date in 'YYYY-MM-DD' format

        Returns:
        --------
        pd.DataFrame
            DataFrame with Date index and OHLCV columns

        Notes:
        ------
        - Uses actual API data only, no synthetic fallbacks
        - Handles BTC-USD special case for crypto ticker format
        """
        print(f"Fetching data for {ticker}...")
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) == 0:
                print(f"WARNING: No data available for {ticker}")
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)
            df.sort_index(inplace=True)
            df = df[["open", "high", "low", "close", "volume"]]
            df.columns = ["Open", "High", "Low", "Close", "Volume"]
            print(f"Successfully fetched {len(df)} records for {ticker}")
            return df
        except Exception as e:
            print(f"ERROR fetching {ticker}: {str(e)}")
            return None

    def fetch_multiple_tickers(self, tickers, start_date, end_date):
        """
        Fetch data for multiple tickers and combine into panel structure

        Returns:
        --------
        dict of pd.DataFrame
            Dictionary with 'Open' and 'Close' DataFrames containing all tickers
        """
        all_data = {}
        for ticker in tickers:
            df = self.fetch_historical_data(ticker, start_date, end_date)
            if df is not None:
                all_data[ticker] = df
        if len(all_data) == 0:
            raise ValueError("No data was successfully fetched for any ticker")
        open_prices = pd.DataFrame({ticker: df["Open"] for ticker, df in all_data.items()})
        close_prices = pd.DataFrame({ticker: df["Close"] for ticker, df in all_data.items()})
        open_prices = open_prices.dropna()
        close_prices = close_prices.dropna()
        print(f"\nSuccessfully combined data for {len(all_data)} tickers")
        print(f"Date range: {open_prices.index[0]} to {open_prices.index[-1]}")
        print(f"Total trading days: {len(open_prices)}")
        return {"open": open_prices, "close": close_prices}


class MaxSharpeOptimizer:
    """
    Maximum Sharpe Ratio Portfolio Optimizer

    Classic mean-variance optimization that maximizes the Sharpe ratio
    subject to weight constraints.
    """

    def __init__(self, risk_free_rate=0.02, max_weight=0.3, min_weight=0.0):
        self.risk_free_rate = risk_free_rate
        self.max_weight = max_weight
        self.min_weight = min_weight

    def optimize(self, returns):
        """
        Find portfolio weights that maximize Sharpe ratio

        Parameters:
        -----------
        returns : pd.DataFrame
            Historical returns (T-1 to T perspective, no look-ahead)

        Returns:
        --------
        np.array
            Optimal portfolio weights
        """
        n_assets = returns.shape[1]
        mean_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252

        def neg_sharpe(weights):
            port_return = np.dot(weights, mean_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            sharpe = (port_return - self.risk_free_rate) / port_vol
            return -sharpe

        constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
        bounds = tuple(((self.min_weight, self.max_weight) for _ in range(n_assets)))
        init_weights = np.array([1 / n_assets] * n_assets)
        result = minimize(
            neg_sharpe,
            init_weights,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000},
        )
        if not result.success:
            print(f"WARNING: Optimization did not converge. Using equal weights.")
            return init_weights
        return result.x


class HRPOptimizer:
    """
    Hierarchical Risk Parity Portfolio Optimizer

    Uses hierarchical clustering to build a diversified portfolio
    that doesn't rely on matrix inversion (more stable than mean-variance).
    """

    def __init__(self):
        pass

    def optimize(self, returns):
        """
        Calculate HRP weights using hierarchical clustering

        Parameters:
        -----------
        returns : pd.DataFrame
            Historical returns

        Returns:
        --------
        np.array
            HRP portfolio weights
        """
        corr_matrix = returns.corr()
        dist_matrix = np.sqrt(0.5 * (1 - corr_matrix))
        dist_condensed = squareform(dist_matrix, checks=False)
        link = linkage(dist_condensed, method="single")
        sort_ix = self._get_quasi_diag(link)
        weights = pd.Series(1.0, index=returns.columns)
        cluster_items = [returns.columns[i] for i in sort_ix]
        weights = self._get_recursive_bisection(returns, corr_matrix, cluster_items)
        return weights.values

    def _get_quasi_diag(self, link):
        """Get quasi-diagonalization order from linkage matrix"""
        link = link.astype(int)
        sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
        num_items = link[-1, 3]
        while sort_ix.max() >= num_items:
            sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
            df0 = sort_ix[sort_ix >= num_items]
            i = df0.index
            j = df0.values - num_items
            sort_ix[i] = link[j, 0]
            df0 = pd.Series(link[j, 1], index=i + 1)
            sort_ix = pd.concat([sort_ix, df0])
            sort_ix = sort_ix.sort_index()
            sort_ix.index = range(sort_ix.shape[0])
        return sort_ix.tolist()

    def _get_cluster_var(self, returns, corr_matrix, cluster_items):
        """Calculate variance of a cluster"""
        cov_slice = returns[cluster_items].cov()
        weights = self._get_inverse_variance_weights(np.diag(cov_slice))
        cluster_var = np.dot(weights.T, np.dot(cov_slice, weights))
        return cluster_var

    def _get_inverse_variance_weights(self, variances):
        """Calculate inverse variance weights"""
        inv_var = 1.0 / variances
        weights = inv_var / inv_var.sum()
        return weights

    def _get_recursive_bisection(self, returns, corr_matrix, items):
        """Recursive bisection to allocate weights"""
        weights = pd.Series(1.0, index=items)
        if len(items) == 1:
            return weights
        split_point = len(items) // 2
        cluster1 = items[:split_point]
        cluster2 = items[split_point:]
        var1 = self._get_cluster_var(returns, corr_matrix, cluster1)
        var2 = self._get_cluster_var(returns, corr_matrix, cluster2)
        alpha = 1 - var1 / (var1 + var2)
        weights[cluster1] *= alpha
        weights[cluster2] *= 1 - alpha
        if len(cluster1) > 1:
            sub_weights1 = self._get_recursive_bisection(returns, corr_matrix, cluster1)
            for item in cluster1:
                weights[item] *= sub_weights1[item]
        if len(cluster2) > 1:
            sub_weights2 = self._get_recursive_bisection(returns, corr_matrix, cluster2)
            for item in cluster2:
                weights[item] *= sub_weights2[item]
        return weights


class MSTOptimizer:
    """
    Minimum Spanning Tree Portfolio Optimizer

    Uses graph theory (MST) to identify asset relationships and
    build a diversified portfolio based on the tree structure.
    """

    def __init__(self):
        pass

    def optimize(self, returns):
        """
        Calculate MST-based weights

        Parameters:
        -----------
        returns : pd.DataFrame
            Historical returns

        Returns:
        --------
        np.array
            MST portfolio weights
        """
        corr_matrix = returns.corr()
        dist_matrix = 1 - corr_matrix.abs()
        mst = self._build_mst(dist_matrix)
        centrality = self._calculate_centrality(mst)
        inv_centrality = 1.0 / (centrality + 1e-06)
        weights = inv_centrality / inv_centrality.sum()
        return weights.values

    def _build_mst(self, dist_matrix):
        """
        Build Minimum Spanning Tree using distance matrix

        Returns adjacency matrix representation of MST
        """
        n = len(dist_matrix)
        assets = dist_matrix.index.tolist()
        mst = pd.DataFrame(0, index=assets, columns=assets)
        edges = []
        for i in range(n):
            for j in range(i + 1, n):
                edges.append((dist_matrix.iloc[i, j], assets[i], assets[j]))
        edges.sort()
        parent = {asset: asset for asset in assets}

        def find(x):
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x, y):
            px, py = (find(x), find(y))
            if px != py:
                parent[px] = py
                return True
            return False

        for dist, u, v in edges:
            if union(u, v):
                mst.loc[u, v] = 1
                mst.loc[v, u] = 1
        return mst

    def _calculate_centrality(self, mst):
        """
        Calculate degree centrality (number of connections) for each node

        Higher centrality = more connected = potentially more correlated
        """
        centrality = mst.sum(axis=1)
        return centrality


class Backtester:
    """
    Backtesting engine that simulates trading with realistic execution

    Key features:
    - Signal at Close[t], Execute at Open[t+1]
    - Portfolio valued at Open[t+1]
    - Returns = Open[t+2]/Open[t+1] - 1
    - Transaction costs applied
    - No look-ahead bias
    """

    def __init__(self, optimizer, price_data, config):
        """
        Parameters:
        -----------
        optimizer : object
            Portfolio optimizer (MaxSharpe, HRP, or MST)
        price_data : dict
            Dictionary with 'open' and 'close' price DataFrames
        config : Config
            Configuration object with all parameters
        """
        self.optimizer = optimizer
        self.open_prices = price_data["open"]
        self.close_prices = price_data["close"]
        self.config = config

    def run_backtest(self, start_date, end_date, lookback_days=252):
        """
        Run backtest for specified period

        Parameters:
        -----------
        start_date : str
            Backtest start date
        end_date : str
            Backtest end date
        lookback_days : int
            Number of days to look back for optimization

        Returns:
        --------
        pd.DataFrame
            Backtest results with portfolio value, weights, returns, etc.
        """
        close_bt = self.close_prices[start_date:end_date]
        open_bt = self.open_prices[start_date:end_date]
        results = []
        portfolio_value = self.config.INITIAL_CAPITAL
        current_weights = None
        previous_shares = None
        rebalance_dates = self._get_rebalance_dates(close_bt, self.config.REBALANCE_FREQUENCY)
        print(f"\nRunning backtest from {start_date} to {end_date}")
        print(
            f"Rebalancing {self.config.REBALANCE_FREQUENCY}ly - {len(rebalance_dates)} rebalances"
        )
        for i, date in enumerate(open_bt.index):
            if i == 0 or date in rebalance_dates:
                if i == 0:
                    hist_start_idx = max(0, i - lookback_days)
                else:
                    hist_start_idx = max(0, i - 1 - lookback_days)
                hist_end_idx = i - 1 if i > 0 else 0
                if hist_end_idx >= lookback_days:
                    hist_closes = close_bt.iloc[hist_start_idx : hist_end_idx + 1]
                    hist_returns = hist_closes.pct_change().dropna()
                    if len(hist_returns) >= 20:
                        new_weights = self.optimizer.optimize(hist_returns)
                        open_prices_today = open_bt.iloc[i]
                        new_shares = portfolio_value * new_weights / open_prices_today
                        if previous_shares is not None:
                            trades = np.abs(new_shares - previous_shares) * open_prices_today
                            transaction_cost = trades.sum() * self.config.COMMISSION_RATE
                            portfolio_value -= transaction_cost
                            new_shares = portfolio_value * new_weights / open_prices_today
                        current_weights = new_weights
                        previous_shares = new_shares
                    elif current_weights is None:
                        current_weights = np.array(
                            [1 / len(open_bt.columns)] * len(open_bt.columns)
                        )
                        open_prices_today = open_bt.iloc[i]
                        previous_shares = portfolio_value * current_weights / open_prices_today
            if current_weights is not None and previous_shares is not None:
                open_prices_today = open_bt.iloc[i]
                portfolio_value = (previous_shares * open_prices_today).sum()
            results.append(
                {
                    "Date": date,
                    "Portfolio_Value": portfolio_value,
                    **{
                        f"Weight_{ticker}": w
                        for ticker, w in zip(
                            open_bt.columns,
                            (
                                current_weights
                                if current_weights is not None
                                else [0] * len(open_bt.columns)
                            ),
                        )
                    },
                }
            )
        results_df = pd.DataFrame(results)
        results_df.set_index("Date", inplace=True)
        results_df["Returns"] = results_df["Portfolio_Value"].pct_change()
        return results_df

    def _get_rebalance_dates(self, price_data, frequency):
        """
        Get rebalance dates based on frequency

        Parameters:
        -----------
        price_data : pd.DataFrame
            Price data with datetime index
        frequency : str
            'M' for monthly, 'Q' for quarterly, 'Y' for yearly

        Returns:
        --------
        list
            List of rebalance dates
        """
        if frequency == "M":
            return price_data.resample("ME").last().index.tolist()
        elif frequency == "Q":
            return price_data.resample("QE").last().index.tolist()
        elif frequency == "Y":
            return price_data.resample("YE").last().index.tolist()
        else:
            raise ValueError(f"Unknown frequency: {frequency}")


class PerformanceAnalyzer:
    """Calculate and display performance metrics"""

    @staticmethod
    def calculate_metrics(returns, risk_free_rate=0.02):
        """
        Calculate comprehensive performance metrics

        Parameters:
        -----------
        returns : pd.Series
            Strategy returns
        risk_free_rate : float
            Annual risk-free rate

        Returns:
        --------
        dict
            Dictionary of performance metrics
        """
        returns = returns.dropna()
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / 252
        annual_return = (1 + total_return) ** (1 / n_years) - 1
        annual_vol = returns.std() * np.sqrt(252)
        sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        downside_returns = returns[returns < 0]
        downside_vol = downside_returns.std() * np.sqrt(252)
        sortino = (annual_return - risk_free_rate) / downside_vol if downside_vol > 0 else 0
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
        return {
            "Total Return": total_return,
            "Annual Return": annual_return,
            "Annual Volatility": annual_vol,
            "Sharpe Ratio": sharpe,
            "Sortino Ratio": sortino,
            "Calmar Ratio": calmar,
            "Max Drawdown": max_drawdown,
            "Win Rate": win_rate,
        }

    @staticmethod
    def print_metrics(metrics, strategy_name):
        """Print metrics in a formatted table"""
        print(f"\n{'=' * 60}")
        print(f"{strategy_name} Performance Metrics")
        print(f"{'=' * 60}")
        print(f"{'Metric':<30} {'Value':>20}")
        print(f"{'-' * 60}")
        for metric, value in metrics.items():
            if (
                "Return" in metric
                or "Volatility" in metric
                or "Drawdown" in metric
                or ("Rate" in metric)
            ):
                print(f"{metric:<30} {value:>19.2%}")
            else:
                print(f"{metric:<30} {value:>20.2f}")
        print(f"{'=' * 60}")


class Visualizer:
    """Create visualizations for backtest results"""

    @staticmethod
    def plot_equity_curves(results_dict, benchmark_results, config):
        """
        Plot equity curves for all strategies vs benchmark

        Parameters:
        -----------
        results_dict : dict
            Dictionary with strategy names as keys and results DataFrames as values
        benchmark_results : pd.DataFrame
            Benchmark results
        config : Config
            Configuration object
        """
        plt.style.use(config.STYLE)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=config.FIGSIZE, dpi=config.DPI)
        for strategy_name, results in results_dict.items():
            normalized = results["Portfolio_Value"] / config.INITIAL_CAPITAL
            ax1.plot(results.index, normalized, label=strategy_name, linewidth=2)
        benchmark_normalized = benchmark_results["Portfolio_Value"] / config.INITIAL_CAPITAL
        ax1.plot(
            benchmark_results.index,
            benchmark_normalized,
            label=config.BENCHMARK_TICKER,
            linewidth=2,
            linestyle="--",
            color="black",
            alpha=0.7,
        )
        ax1.set_title("Portfolio Equity Curves (Normalized to 1.0)", fontsize=14, fontweight="bold")
        ax1.set_ylabel("Portfolio Value (Normalized)", fontsize=12)
        ax1.legend(loc="best", fontsize=10)
        ax1.grid(True, alpha=0.3)
        for strategy_name, results in results_dict.items():
            cumulative = results["Portfolio_Value"] / config.INITIAL_CAPITAL
            running_max = cumulative.expanding().max()
            drawdown = (cumulative - running_max) / running_max
            ax2.fill_between(results.index, drawdown * 100, 0, alpha=0.3, label=strategy_name)
        cumulative_bm = benchmark_normalized
        running_max_bm = cumulative_bm.expanding().max()
        drawdown_bm = (cumulative_bm - running_max_bm) / running_max_bm
        ax2.fill_between(
            benchmark_results.index,
            drawdown_bm * 100,
            0,
            alpha=0.2,
            color="black",
            label=config.BENCHMARK_TICKER,
        )
        ax2.set_title("Drawdown Analysis", fontsize=14, fontweight="bold")
        ax2.set_xlabel("Date", fontsize=12)
        ax2.set_ylabel("Drawdown (%)", fontsize=12)
        ax2.legend(loc="best", fontsize=10)
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_weights_evolution(results, strategy_name, config):
        """
        Plot evolution of portfolio weights over time

        Parameters:
        -----------
        results : pd.DataFrame
            Backtest results with weight columns
        strategy_name : str
            Name of the strategy
        config : Config
            Configuration object
        """
        plt.style.use(config.STYLE)
        fig, ax = plt.subplots(figsize=config.FIGSIZE, dpi=config.DPI)
        weight_cols = [col for col in results.columns if col.startswith("Weight_")]
        weights_df = results[weight_cols]
        weights_df.columns = [col.replace("Weight_", "") for col in weight_cols]
        ax.stackplot(
            results.index,
            *[weights_df[col] * 100 for col in weights_df.columns],
            labels=weights_df.columns,
            alpha=0.7,
        )
        ax.set_title(
            f"{strategy_name} - Portfolio Weights Evolution", fontsize=14, fontweight="bold"
        )
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Weight (%)", fontsize=12)
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 100])
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_returns_distribution(results_dict, benchmark_results, config):
        """
        Plot returns distribution for all strategies

        Parameters:
        -----------
        results_dict : dict
            Dictionary with strategy results
        benchmark_results : pd.DataFrame
            Benchmark results
        config : Config
            Configuration object
        """
        plt.style.use(config.STYLE)
        fig, axes = plt.subplots(2, 2, figsize=config.FIGSIZE, dpi=config.DPI)
        axes = axes.flatten()
        strategies = list(results_dict.keys()) + [config.BENCHMARK_TICKER]
        all_results = list(results_dict.values()) + [benchmark_results]
        for idx, (strategy_name, results) in enumerate(zip(strategies, all_results)):
            if idx < len(axes):
                returns = results["Returns"].dropna() * 100
                axes[idx].hist(returns, bins=50, alpha=0.7, edgecolor="black")
                axes[idx].axvline(
                    returns.mean(),
                    color="red",
                    linestyle="--",
                    linewidth=2,
                    label=f"Mean: {returns.mean():.2f}%",
                )
                axes[idx].axvline(0, color="black", linestyle="-", linewidth=1)
                axes[idx].set_title(
                    f"{strategy_name} Returns Distribution", fontsize=12, fontweight="bold"
                )
                axes[idx].set_xlabel("Daily Returns (%)", fontsize=10)
                axes[idx].set_ylabel("Frequency", fontsize=10)
                axes[idx].legend(fontsize=9)
                axes[idx].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


def main():
    """Main execution function"""
    print("=" * 80)
    print("PORTFOLIO OPTIMIZATION STRATEGIES: MAX SHARPE, HRP, MST")
    print("=" * 80)
    config = Config()
    print("\nConfiguration:")
    print(f"  Tickers: {', '.join(config.TICKERS)}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")
    print(f"  In-Sample: {config.IS_START} to {config.IS_END}")
    print(f"  Out-of-Sample: {config.OOS_START} to {config.OOS_END}")
    print(f"  Rebalancing: {config.REBALANCE_FREQUENCY}ly")
    print(f"  Commission: {config.COMMISSION_RATE * 100:.1f} bps per trade")
    print("\n" + "=" * 80)
    print("STEP 1: FETCHING DATA FROM Yahoo Finance")
    print("=" * 80)
    fetcher = DataFetcher()
    extended_start = "2012-01-01"
    data = fetcher.fetch_multiple_tickers(config.TICKERS, extended_start, config.OOS_END)
    print("\nFetching benchmark data...")
    benchmark_data = fetcher.fetch_multiple_tickers(
        [config.BENCHMARK_TICKER], extended_start, config.OOS_END
    )
    print("\n" + "=" * 80)
    print("STEP 2: INITIALIZING OPTIMIZERS")
    print("=" * 80)
    optimizers = {
        "Max Sharpe": MaxSharpeOptimizer(
            risk_free_rate=config.RISK_FREE_RATE,
            max_weight=config.MAX_WEIGHT,
            min_weight=config.MIN_WEIGHT,
        ),
        "HRP": HRPOptimizer(),
        "MST": MSTOptimizer(),
    }
    print(f"Initialized {len(optimizers)} optimizers: {', '.join(optimizers.keys())}")
    print("\n" + "=" * 80)
    print("STEP 3: RUNNING BACKTESTS")
    print("=" * 80)
    is_results = {}
    for strategy_name, optimizer in optimizers.items():
        print(f"\n--- {strategy_name} (In-Sample) ---")
        backtester = Backtester(optimizer, data, config)
        results = backtester.run_backtest(
            config.IS_START, config.IS_END, lookback_days=config.LOOKBACK_DAYS
        )
        is_results[strategy_name] = results
    oos_results = {}
    for strategy_name, optimizer in optimizers.items():
        print(f"\n--- {strategy_name} (Out-of-Sample) ---")
        backtester = Backtester(optimizer, data, config)
        results = backtester.run_backtest(
            config.OOS_START, config.OOS_END, lookback_days=config.LOOKBACK_DAYS
        )
        oos_results[strategy_name] = results
    print("\n--- Benchmark (In-Sample) ---")
    bm_is_prices = (
        benchmark_data["open"].loc[config.IS_START : config.IS_END, config.BENCHMARK_TICKER].copy()
    )
    bm_is_results = pd.DataFrame(
        {
            "Portfolio_Value": bm_is_prices / bm_is_prices.iloc[0] * config.INITIAL_CAPITAL,
            "Returns": bm_is_prices.pct_change(),
        },
        index=bm_is_prices.index,
    )
    print("\n--- Benchmark (Out-of-Sample) ---")
    bm_oos_prices = (
        benchmark_data["open"]
        .loc[config.OOS_START : config.OOS_END, config.BENCHMARK_TICKER]
        .copy()
    )
    bm_oos_results = pd.DataFrame(
        {
            "Portfolio_Value": bm_oos_prices / bm_oos_prices.iloc[0] * config.INITIAL_CAPITAL,
            "Returns": bm_oos_prices.pct_change(),
        },
        index=bm_oos_prices.index,
    )
    print("\n" + "=" * 80)
    print("STEP 4: PERFORMANCE ANALYSIS")
    print("=" * 80)
    analyzer = PerformanceAnalyzer()
    print("\n" + "=" * 80)
    print("IN-SAMPLE PERFORMANCE (2013-2019)")
    print("=" * 80)
    is_metrics = {}
    for strategy_name, results in is_results.items():
        metrics = analyzer.calculate_metrics(results["Returns"], config.RISK_FREE_RATE)
        is_metrics[strategy_name] = metrics
        analyzer.print_metrics(metrics, f"{strategy_name} (IS)")
    bm_is_metrics = analyzer.calculate_metrics(bm_is_results["Returns"], config.RISK_FREE_RATE)
    analyzer.print_metrics(bm_is_metrics, f"{config.BENCHMARK_TICKER} Benchmark (IS)")
    print("\n" + "=" * 80)
    print("OUT-OF-SAMPLE PERFORMANCE (2020-2023)")
    print("=" * 80)
    oos_metrics = {}
    for strategy_name, results in oos_results.items():
        metrics = analyzer.calculate_metrics(results["Returns"], config.RISK_FREE_RATE)
        oos_metrics[strategy_name] = metrics
        analyzer.print_metrics(metrics, f"{strategy_name} (OOS)")
    bm_oos_metrics = analyzer.calculate_metrics(bm_oos_results["Returns"], config.RISK_FREE_RATE)
    analyzer.print_metrics(bm_oos_metrics, f"{config.BENCHMARK_TICKER} Benchmark (OOS)")
    print("\n" + "=" * 80)
    print("STEP 5: GENERATING VISUALIZATIONS")
    print("=" * 80)
    visualizer = Visualizer()
    print("\nGenerating In-Sample equity curves...")
    visualizer.plot_equity_curves(is_results, bm_is_results, config)
    print("\nGenerating Out-of-Sample equity curves...")
    visualizer.plot_equity_curves(oos_results, bm_oos_results, config)
    print("\nGenerating returns distributions...")
    visualizer.plot_returns_distribution(oos_results, bm_oos_results, config)
    for strategy_name, results in oos_results.items():
        print(f"\nGenerating weight evolution for {strategy_name}...")
        visualizer.plot_weights_evolution(results, f"{strategy_name} (OOS)", config)
    print("\n" + "=" * 80)
    print("COMPARATIVE SUMMARY")
    print("=" * 80)
    summary_data = []
    for strategy_name in optimizers.keys():
        summary_data.append(
            {
                "Strategy": strategy_name,
                "IS Return": is_metrics[strategy_name]["Annual Return"],
                "IS Sharpe": is_metrics[strategy_name]["Sharpe Ratio"],
                "IS Max DD": is_metrics[strategy_name]["Max Drawdown"],
                "OOS Return": oos_metrics[strategy_name]["Annual Return"],
                "OOS Sharpe": oos_metrics[strategy_name]["Sharpe Ratio"],
                "OOS Max DD": oos_metrics[strategy_name]["Max Drawdown"],
            }
        )
    summary_data.append(
        {
            "Strategy": config.BENCHMARK_TICKER,
            "IS Return": bm_is_metrics["Annual Return"],
            "IS Sharpe": bm_is_metrics["Sharpe Ratio"],
            "IS Max DD": bm_is_metrics["Max Drawdown"],
            "OOS Return": bm_oos_metrics["Annual Return"],
            "OOS Sharpe": bm_oos_metrics["Sharpe Ratio"],
            "OOS Max DD": bm_oos_metrics["Max Drawdown"],
        }
    )
    summary_df = pd.DataFrame(summary_data)
    print("\nPerformance Comparison Table:")
    print("-" * 100)
    print(
        f"{'Strategy':<15} {'IS Return':>12} {'IS Sharpe':>12} {'IS Max DD':>12} {'OOS Return':>12} {'OOS Sharpe':>12} {'OOS Max DD':>12}"
    )
    print("-" * 100)
    for _, row in summary_df.iterrows():
        print(
            f"{row['Strategy']:<15} {row['IS Return']:>11.2%} {row['IS Sharpe']:>12.2f} {row['IS Max DD']:>11.2%} {row['OOS Return']:>11.2%} {row['OOS Sharpe']:>12.2f} {row['OOS Max DD']:>11.2%}"
        )
    print("-" * 100)
    print("\n" + "=" * 80)
    print("BACKTEST COMPLETED SUCCESSFULLY")
    print("=" * 80)
    return {
        "is_results": is_results,
        "oos_results": oos_results,
        "is_metrics": is_metrics,
        "oos_metrics": oos_metrics,
        "benchmark_is": bm_is_results,
        "benchmark_oos": bm_oos_results,
    }


if __name__ == "__main__":
    results = main()
